# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-10 · Turns the ML-08 model and the ML-09
> audit into something a person can actually work down on a Monday morning.
>
> The queue is built on the **sealed frame** (features Mar–May 2026), scored by a model trained
> only on the development window. June 2026 outcomes are used **once**, to report what the queue
> would have been worth — never to tune it.

In [1]:
%pip install -q pandas scikit-learn matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, subprocess, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

SEED = 20260808
REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT = REPO / "work/outputs"; FIG = REPO / "work/figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()

if not (OUT / "modeling_frame_sealed.parquet").exists():
    subprocess.run([sys.executable, str(REPO/"work/scripts/build_modeling_frame.py"), "--sealed"],
                   check=True)

dev    = pd.read_parquet(OUT / "modeling_frame_dev.parquet").reset_index(drop=True)
sealed = pd.read_parquet(OUT / "modeling_frame_sealed.parquet").reset_index(drop=True)

NUM = ["f_pos","f_impressions","f_clicks","f_ctr","f_days_with_impressions","f_pos_volatility",
       "f_pos_trend","search_volume","competition","cpc","backlinks","word_count","char_count",
       "keyword_token_count","content_age_days"]
CAT = ["content_type","main_intent","competition_level"]
FLAGS = ["f_pos_trend","backlinks","word_count","search_volume"]

def make_X(d):
    X = d[NUM + CAT].copy()
    for c in FLAGS: X[f"has_{c}"] = X[c].notna().astype(int)
    return X, [c for c in X.columns if c not in CAT]

X_dev, num = make_X(dev); X_sl, _ = make_X(sealed)
model = Pipeline([("pre", ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
    ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), CAT)])),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                                   random_state=SEED))]).fit(X_dev, dev.is_position_decline.values)

q = sealed.copy()
q["risk"] = model.predict_proba(X_sl)[:, 1]
y_sl = q.is_position_decline.values
print(f"scored {len(q):,} pages · {q.client_hash_id.nunique()} clients · "
      f"observed June decline rate {y_sl.mean():.3f}")

scored 116,656 pages · 48 clients · observed June decline rate 0.562


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue answers one question: **"what should I open first?"** Three things make it usable by a
person rather than by a script.

**1. An action, not a score.** Each row carries a verb. The action is decided by *which* evidence
fired, not by how high the score is:

| Action | Fires when | What the person does |
|---|---|---|
| `review_now` | already sliding **and** on page 1–2 | open the page, check what changed, refresh if warranted |
| `investigate_volatility` | unstable position, no clear direction | look for technical or SERP-layout causes before touching content |
| `defend_position` | high visibility, currently stable, shallow | leave the content alone; watch weekly |
| `monitor` | everything else | no action this cycle |

**2. A per-client cap — the fix ML-08 said was needed.** Concentration was severe on the development
window (one client took 62% of the model's top 50, and 78% under the client-relative variant). On this
sealed window it is milder but still real: **38% of the uncapped top 50 belongs to one client, across
just 6 clients**. Either way a queue is worked by people who own portfolios, and "your 50 most urgent
pages" that are mostly someone else's account is not a queue. I cap each client at **5 pages per 50**
and measure what the cap costs rather than assuming it is free.

**3. Reason codes in plain words.** Every row states its evidence in the same vocabulary the
ML-07 rule used, so a reviewer can disagree with a specific claim rather than with "the model".

In [3]:
sliding   = q.f_pos_trend.fillna(0) >= 1.0
shallow   = (q.f_pos > 0) & (q.f_pos <= 20)
unstable  = q.f_pos_volatility.fillna(0) >= q.f_pos_volatility.median()
visible   = q.f_impressions >= 1000

q["action"] = np.select(
    [sliding & shallow, unstable & ~sliding, visible & shallow & ~sliding & ~unstable],
    ["review_now", "investigate_volatility", "defend_position"], default="monitor")

codes = {"already_sliding": sliding, "shallow_and_exposed": shallow,
         "unstable_position": unstable, "high_visibility": visible,
         "intermittent_visibility": q.f_days_with_impressions < 60}
q["reason_codes"] = [",".join(k for k, v in codes.items() if v.iloc[i]) or "none"
                     for i in range(len(q))]
q["confidence"] = pd.cut(q.risk, [0, .55, .75, 1.0],
                         labels=["low", "medium", "high"], include_lowest=True)

def capped_queue(d, per_client=5, n=50):
    """Rank by risk, but let no client take more than `per_client` of the top n."""
    out, taken = [], {}
    for i in d.sort_values("risk", ascending=False).index:
        c = d.at[i, "client_hash_id"]
        if taken.get(c, 0) >= per_client: continue
        out.append(i); taken[c] = taken.get(c, 0) + 1
        if len(out) == n: break
    return d.loc[out]

top_raw = q.nlargest(50, "risk")
top_cap = capped_queue(q, per_client=5, n=50)

for name, t in [("uncapped", top_raw), ("capped at 5/client", top_cap)]:
    share = t.client_hash_id.value_counts().iloc[0] / len(t)
    print(f"{name:20s} precision@50 = {t.is_position_decline.mean():.3f} · "
          f"largest client {share:.0%} · {t.client_hash_id.nunique()} distinct clients")
print(f"\nobserved base rate {y_sl.mean():.3f}")
print("\nAction mix across the whole scored population:")
print(q.action.value_counts().to_string())

uncapped             precision@50 = 0.860 · largest client 38% · 6 distinct clients
capped at 5/client   precision@50 = 0.760 · largest client 10% · 12 distinct clients

observed base rate 0.562

Action mix across the whole scored population:
action
review_now                53978
monitor                   27834
investigate_volatility    20622
defend_position           14222


In [4]:
top_cap = top_cap.assign(page=["p"+h[-6:] for h in top_cap.content_hash_id],
                         client=["c"+h[-4:] for h in top_cap.client_hash_id])
print("THE QUEUE — top 20 after the per-client cap\n")
print(top_cap.head(20)[["page","client","action","confidence","risk","f_pos","f_pos_trend",
                        "f_impressions","reason_codes"]].to_string(
    index=False, formatters={"risk":"{:.2f}".format, "f_pos":"{:.1f}".format,
                             "f_pos_trend":"{:+.1f}".format, "f_impressions":"{:,.0f}".format}))

THE QUEUE — top 20 after the per-client cap

   page client     action confidence risk f_pos f_pos_trend f_impressions                                                          reason_codes
p0817a2  c8636 review_now       high 0.98  12.5       +20.0         2,573 already_sliding,shallow_and_exposed,unstable_position,high_visibility
p1df31e  cb4db review_now       high 0.98  10.7        +9.5         3,202 already_sliding,shallow_and_exposed,unstable_position,high_visibility
p50b4d5  c8636 review_now       high 0.98  19.2        +9.5         2,449 already_sliding,shallow_and_exposed,unstable_position,high_visibility
pc713cf  c8636 review_now       high 0.98  15.2       +12.1         1,358 already_sliding,shallow_and_exposed,unstable_position,high_visibility
p32cd1f  c3229 review_now       high 0.98   7.6       +14.5         1,628 already_sliding,shallow_and_exposed,unstable_position,high_visibility
p75117c  c8636 review_now       high 0.97   8.5       +11.7         2,586 already_sliding,s

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who this is for, and for what

**User:** an SEO or content strategist who owns a portfolio and has time to review perhaps 20–50
pages in a cycle. **Decision it supports:** the *order* of a manual review queue — nothing else.
**Cost of a wrong call:** an hour spent opening a page that turned out fine. That asymmetry is
why this ships as a queue and not as an automation: the downside of a false positive is an hour,
the downside of an automated content change is a live site.

### Where it stops being valid

- **It ranks; it does not predict a page's fate.** The scores are ordered risk, not calibrated
  probabilities — ML-08 measured them as over-confident at the top (predicting ~0.84 where ~0.80
  occurred). "High confidence" means *more evidence agreed*, not *84% likely*.
- **It says nothing about cause.** No page in this data was randomly assigned a refresh. The
  queue cannot tell you that reviewing a page will change its trajectory — only that pages
  matching this evidence pattern were *observed* to decline more often.
- **Position is not traffic and not revenue.** The label is impression-weighted average position.
  A page can lose average position while gaining clicks, if it slipped on low-value queries and
  held the ones that convert. Nothing here measures query value or money.
- **It only covers measurable pages.** Eligibility required ≥100 impressions in the feature
  window and ≥30 in the outcome month. Pages with little or no search visibility are absent —
  and a page that vanished completely looks the same as a page that was never eligible.
- **It does not work equally for every client.** ML-09 measured per-client AUC from 0.48 to 0.77.
  For at least one portfolio it is no better than a coin flip. Per-client performance should be
  reported alongside any deployment, and the queue should abstain where it has no measured skill.
- **It has a shelf life.** Features come from a 90-day window; the model was fitted on Jan–Mar
  2026 and validated on June 2026. It has never been tested more than three months past its
  training window.

In [5]:
# The limits above are measurable, so measure them rather than only asserting them.
print("Coverage — who is NOT in this queue:")
print(f"  scored pages                    : {len(q):,}")
print(f"  clients represented             : {q.client_hash_id.nunique()} "
      f"(of 104 in the warehouse dim_clients)")
print(f"  pages excluded by the volume floor is by construction: the frame only contains")
print(f"  pages with >=100 feature-window and >=30 outcome-window impressions.")

print("\nCalibration on the sealed frame — are the confidence bands honest?")
print(q.groupby("confidence", observed=True).agg(
    pages=("risk","size"), mean_risk=("risk","mean"),
    observed_decline=("is_position_decline","mean")).round(3).to_string())
print("\nThe bands are ordered correctly, which is what a queue needs. They are not")
print("probabilities, which is why the playbook never quotes them as percentages.")

Coverage — who is NOT in this queue:
  scored pages                    : 116,656
  clients represented             : 48 (of 104 in the warehouse dim_clients)
  pages excluded by the volume floor is by construction: the frame only contains
  pages with >=100 feature-window and >=30 outcome-window impressions.

Calibration on the sealed frame — are the confidence bands honest?
            pages  mean_risk  observed_decline
confidence                                    
low         36497      0.422             0.374
medium      42741      0.653             0.544
high        37418      0.829             0.766

The bands are ordered correctly, which is what a queue needs. They are not
probabilities, which is why the playbook never quotes them as percentages.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### What a person must check before acting

For every page the queue surfaces, before touching anything:

1. **Did the page actually change, or did the SERP?** A position slide with stable impressions
   often means a competitor moved or Google changed the layout — not that the page got worse.
2. **Is the decline in queries you care about?** Pull the page's query mix. Losing position on
   irrelevant long-tail terms while holding the money terms is not a problem to fix.
3. **Is there a technical cause?** Indexation, canonicalisation, a redirect, a template change,
   Core Web Vitals. Content refreshes cannot fix technical problems, and the model cannot see them.
4. **Was the page deliberately changed recently?** A drop after an intentional edit is a
   different conversation from an unexplained drift.
5. **Does the reason code match what you see?** The codes are the model's argument. If
   `already_sliding` fired but the page looks stable in your own tooling, trust your tooling and
   tell whoever maintains the queue.

### The no-go list — what must never be automated off this score

- **Never auto-publish or auto-rewrite content** from this queue. It ranks risk; it has no
  measure of content quality and no evidence that rewriting helps.
- **Never auto-deindex, delete, or redirect** a page. A low score means "no evidence of risk",
  which is not the same as "worthless".
- **Never use it for client-facing performance reporting or billing.** Per-client skill varies
  from coin-flip to useful; a number that unreliable must not reach an invoice or a QBR deck.
- **Never present the score as a probability** ("this page has an 86% chance of dropping").
- **Never let it justify a causal claim** — not in a report, not in a pitch, not in a headline.
- **Never run it on a client whose per-client AUC has not been measured.** Unmeasured is not
  the same as fine.

In [6]:
# A deployment gate, as code: refuse to rank clients where skill was never established.
from sklearn.metrics import roc_auc_score
per_client = (q.groupby("client_hash_id")
               .filter(lambda g: len(g) >= 300 and g.is_position_decline.nunique() > 1)
               .groupby("client_hash_id")
               .apply(lambda g: pd.Series({"pages": len(g),
                                           "auc": roc_auc_score(g.is_position_decline, g.risk)}),
                      include_groups=False))
MIN_AUC = 0.60
per_client["gate"] = np.where(per_client.auc >= MIN_AUC, "serve", "ABSTAIN")
per_client.index = ["c"+i[-4:] for i in per_client.index]
print(f"Per-client gate at AUC >= {MIN_AUC} (clients with >=300 scored pages):\n")
print(per_client.sort_values("auc").round(3).to_string())
n_abstain = (per_client.gate == "ABSTAIN").sum()
print(f"\n{n_abstain} of {len(per_client)} measured clients would be served NO queue this cycle.")
print("Abstaining is a feature. A queue that is wrong for a portfolio is worse than no queue.")

Per-client gate at AUC >= 0.6 (clients with >=300 scored pages):

         pages    auc     gate
ce525    468.0  0.538  ABSTAIN
c5e64    495.0  0.547  ABSTAIN
c9543    449.0  0.564  ABSTAIN
c8efc    908.0  0.586  ABSTAIN
c63c4  13361.0  0.596  ABSTAIN
cffd8   3431.0  0.597  ABSTAIN
cd1de   2652.0  0.601    serve
ca4a0   1680.0  0.603    serve
c81d4   4434.0  0.613    serve
c0096  17400.0  0.628    serve
c11d5   1836.0  0.633    serve
cf715   1669.0  0.636    serve
cf586   1934.0  0.641    serve
cf01b    623.0  0.646    serve
c7c86    481.0  0.648    serve
c1abf   2857.0  0.652    serve
c46ef    932.0  0.660    serve
c5515   1591.0  0.661    serve
c4f3d   2725.0  0.688    serve
cf3b6   1117.0  0.695    serve
c62c0   8562.0  0.701    serve
cb4db    796.0  0.703    serve
c3229   8696.0  0.708    serve
ca37d    300.0  0.718    serve
c8242   4867.0  0.750    serve
c7cbb    690.0  0.756    serve
c8636   5253.0  0.759    serve
c65ea  25239.0  0.764    serve

6 of 28 measured clients would be 

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### What would tell me these recommendations went stale

Four triggers, each with a number attached so it can be checked automatically rather than argued
about:

| Trigger | Threshold | Why it matters | Response |
|---|---|---|---|
| **Base-rate drift** | population decline rate moves outside 0.45–0.65 | the label's meaning has shifted; a 1.4× lift over a 0.57 base is not the same product as over a 0.30 base | rebuild the frame, re-report all lifts |
| **Precision decay** | rolling P@50 on the last completed month drops below **0.65** for two consecutive cycles | the ranking has stopped earning its place above the rule | retrain; if it does not recover, fall back to the ML-07 rule |
| **Per-client collapse** | any served client's AUC drops below 0.60 | skill was portfolio-specific and that portfolio changed | abstain for that client until re-measured |
| **Feature drift** | median `f_pos` or `f_pos_volatility` moves more than 25% from the training window | the input distribution is no longer the one the model was fitted on | investigate before trusting any score |

**Scheduled retrain regardless:** every quarter. The model was fitted on a three-month window and
validated exactly one window forward. It has no demonstrated shelf life beyond that, and the
honest response to "how long is it good for?" is "we have measured three months".

**The fallback is real, not rhetorical.** The ML-07 rule scored P@50 = 0.780 on the sealed frame
against the model's 0.860 — close enough that reverting to a transparent rule costs little and
requires no retraining. Any monitoring trigger that fires can fall back to it the same day.

**Two of these triggers fired on the sealed window, and I am reporting that rather than tuning
the thresholds until they passed.** Median `f_pos` moved from 8.18 in the training window to
10.58 (**+29.4%**), and median `f_pos_volatility` from 6.13 to 7.99 (**+30.4%**) — both past the
25% band. The pages being scored in the sealed window sit measurably deeper and bounce more than
the pages the model was fitted on.

This is consistent with what ML-04 already documented: the panel is unbalanced and growing, from
262k content items in January to 409k in June, and newly-entering pages tend to rank deeper. So
the drift is largely *composition*, not a broken model — which the precision numbers support,
since P@50 held at 0.860 uncapped on that same window.

But "the trigger fired and the metric was fine anyway" is exactly the situation where a team
quietly widens the threshold. The honest response is the one written in the table: **investigate
before trusting any score**, and treat every number in this playbook as provisional until the
model is refitted on a window whose composition matches the one it will score.

In [7]:
# The triggers as a runnable check against the current window.
train_med_pos = dev.f_pos.median(); train_med_vol = dev.f_pos_volatility.median()
checks = [
    ("base rate in [0.45, 0.65]", 0.45 <= y_sl.mean() <= 0.65, f"{y_sl.mean():.3f}"),
    ("P@50 (capped queue) >= 0.65", top_cap.is_position_decline.mean() >= 0.65,
     f"{top_cap.is_position_decline.mean():.3f}"),
    ("median f_pos drift < 25%",
     abs(q.f_pos.median() - train_med_pos) / train_med_pos < 0.25,
     f"train {train_med_pos:.2f} -> now {q.f_pos.median():.2f} "
     f"({(q.f_pos.median()-train_med_pos)/train_med_pos:+.1%})"),
    ("median volatility drift < 25%",
     abs(q.f_pos_volatility.median() - train_med_vol) / train_med_vol < 0.25,
     f"train {train_med_vol:.2f} -> now {q.f_pos_volatility.median():.2f} "
     f"({(q.f_pos_volatility.median()-train_med_vol)/train_med_vol:+.1%})"),
]
print(f"{'trigger':34s} {'state':8s} value")
print("-"*80)
for n, ok, v in checks:
    print(f"{n:34s} {'ok' if ok else 'FIRED':8s} {v}")
if not all(c[1] for c in checks):
    print("\nAt least one trigger fired on the current window — see the response column above.")

trigger                            state    value
--------------------------------------------------------------------------------
base rate in [0.45, 0.65]          ok       0.562
P@50 (capped queue) >= 0.65        ok       0.760
median f_pos drift < 25%           FIRED    train 8.18 -> now 10.58 (+29.4%)
median volatility drift < 25%      FIRED    train 6.13 -> now 7.99 (+30.4%)

At least one trigger fired on the current window — see the response column above.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
QCOLS = ["rank","content_hash_id","client_hash_id","risk","confidence","action","reason_codes",
         "f_pos","f_pos_trend","f_pos_volatility","f_impressions","f_clicks","f_ctr",
         "f_days_with_impressions"]
full = q.sort_values("risk", ascending=False).reset_index(drop=True)
full["rank"] = np.arange(1, len(full)+1)
full[QCOLS].to_csv(OUT/"action_playbook_queue.csv", index=False)
capped = capped_queue(q, 5, 500).reset_index(drop=True)
capped["rank"] = np.arange(1, len(capped)+1)
capped[QCOLS].to_csv(OUT/"action_playbook_queue_capped500.csv", index=False)

# ---- figures for the paper ----
plt.rcParams.update({"figure.dpi":140, "font.size":9, "axes.spines.top":False,
                     "axes.spines.right":False})

ks = [10,25,50,100,250,500,1000,2500,5000]
def patk(s,yy,k): return np.asarray(yy)[np.argsort(-np.asarray(s),kind="stable")[:k]].mean()
rule_pts = ((q.f_pos_trend.fillna(0)>=1.0).astype(int)*3
            + (q.f_pos_volatility.fillna(0)>=q.f_pos_volatility.median()).astype(int)
            + ((q.f_pos>0)&(q.f_pos<=20)).astype(int) + (q.f_impressions>=1000).astype(int)
            + (q.f_days_with_impressions<60).astype(int)).values
e = np.log1p(q.f_impressions.values); rule_score = rule_pts + 0.999*(e-e.min())/(e.max()-e.min())

fig, ax = plt.subplots(figsize=(6,3.6))
ax.plot(ks, [patk(q.risk,y_sl,k) for k in ks], "o-", label="random forest")
ax.plot(ks, [patk(rule_score,y_sl,k) for k in ks], "s--", label="ML-07 rule (frozen)")
ax.axhline(y_sl.mean(), color="grey", ls=":", label=f"base rate ({y_sl.mean():.2f})")
ax.set_xscale("log"); ax.set_xlabel("K (queue depth)"); ax.set_ylabel("precision@K")
ax.set_title("Sealed June 2026: precision at queue depth"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG/"precision_at_k_sealed.svg"); plt.close(fig)

fig, ax = plt.subplots(figsize=(6,3.2))
ac = q.action.value_counts()
ar = q.groupby("action").is_position_decline.mean().reindex(ac.index)
ax.barh(range(len(ac)), ac.values, color="#4C78A8")
ax.set_yticks(range(len(ac)))
ax.set_yticklabels([f"{a}\n(observed decline {r:.0%})" for a, r in zip(ac.index, ar.values)])
ax.set_xlabel("pages"); ax.set_title("Recommended action mix, sealed window")
ax.invert_yaxis(); fig.tight_layout(); fig.savefig(FIG/"action_mix.svg"); plt.close(fig)

fig, ax = plt.subplots(figsize=(6,3.2))
for lbl, d in [("uncapped", q.nlargest(200,"risk")), ("capped 5/client", capped_queue(q,5,200))]:
    vc = d.client_hash_id.value_counts(normalize=True).cumsum().reset_index(drop=True)
    ax.plot(range(1,len(vc)+1), vc.values, marker="o", ms=3, label=lbl)
ax.set_xlabel("clients (ranked by share of queue)"); ax.set_ylabel("cumulative share of top 200")
ax.set_title("Client concentration in the queue"); ax.legend(frameon=False); ax.set_xlim(0,15)
fig.tight_layout(); fig.savefig(FIG/"client_concentration.svg"); plt.close(fig)

json.dump({
 "seed": SEED, "scored_pages": int(len(q)), "clients": int(q.client_hash_id.nunique()),
 "sealed_base_rate": round(float(y_sl.mean()),4),
 "precision_at_k_model": {str(k): round(float(patk(q.risk,y_sl,k)),4) for k in ks},
 "precision_at_k_rule":  {str(k): round(float(patk(rule_score,y_sl,k)),4) for k in ks},
 "top50_uncapped": {"precision": round(float(top_raw.is_position_decline.mean()),4),
   "largest_client_share": round(float(top_raw.client_hash_id.value_counts().iloc[0]/50),4),
   "distinct_clients": int(top_raw.client_hash_id.nunique())},
 "top50_capped": {"precision": round(float(top_cap.is_position_decline.mean()),4),
   "largest_client_share": round(float(top_cap.client_hash_id.value_counts().iloc[0]/50),4),
   "distinct_clients": int(top_cap.client_hash_id.nunique())},
 "action_mix": {k:int(v) for k,v in q.action.value_counts().items()},
 "clients_gated_out": int(n_abstain),
}, open(OUT/"w07_playbook_metrics.json","w"), indent=2)

for f in ["action_playbook_queue.csv","action_playbook_queue_capped500.csv","w07_playbook_metrics.json"]:
    print("wrote", rel(OUT/f))
for f in ["precision_at_k_sealed.svg","action_mix.svg","client_concentration.svg"]:
    print("wrote", rel(FIG/f))

wrote work/outputs/action_playbook_queue.csv
wrote work/outputs/action_playbook_queue_capped500.csv
wrote work/outputs/w07_playbook_metrics.json
wrote work/figures/precision_at_k_sealed.svg
wrote work/figures/action_mix.svg
wrote work/figures/client_concentration.svg


**What the paper builds on.** `work/outputs/` now holds the full ranked queue, the capped queue,
and `w07_playbook_metrics.json` — the receipts every number in the write-up traces back to. The
CSVs are gitignored by design (they are data); the metrics JSON is committed. Three figures land
in `work/figures/`: precision at queue depth against the frozen rule and the base rate, the
action mix with observed decline rates, and the client-concentration curve with and without the
cap.

**The one-sentence version for the paper:** *in this pseudonymized portfolio, ordering a manual
review queue by observed ranking-momentum signals surfaced declining pages at roughly 1.5× the
base rate on a sealed future month, with a transparent hand-written rule capturing most of that
gain — which makes this a modest, decision-support-grade result, not a breakthrough.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.